# 02 – Regressione: il prezzo delle case

In questo notebook:
1. generiamo dei dati di case, con qualche errore dentro;
2. li puliamo;
3. troviamo la retta migliore in due modi (formula e discesa del gradiente);
4. misuriamo l'errore su dati mai visti;
5. vediamo l'overfitting.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

## 1. I dati

Creiamo 200 case finte: metri quadri, stanze e prezzo (in migliaia di euro).
Poi, come nella realtà, ci mettiamo dentro qualche errore.

In [ ]:
rng = np.random.default_rng(42)
n = 200
metri = rng.uniform(40, 200, n).round()
stanze = np.clip((metri / 30 + rng.normal(0, 0.7, n)).round(), 1, 6)
prezzo = (20 + 1.6 * metri + 8 * stanze + rng.normal(0, 18, n)).round(1)

case = pd.DataFrame({'metri': metri, 'stanze': stanze, 'prezzo': prezzo})

# errori tipici
case.loc[5, 'prezzo'] = 0.001          # prezzo scritto male
case.loc[17, 'stanze'] = np.nan        # dato mancante
case.loc[33, 'metri'] = np.nan
case.loc[48, 'metri'] = 1450           # metri in piedi quadri invece che m²

case.head(10)

In [ ]:
case.describe()

**Domanda:** guardando `describe()`, quali numeri non tornano?

## 2. Pulizia

*Garbage in, garbage out*: con dati sporchi anche il modello migliore sbaglia.

In [ ]:
print('Righe con valori mancanti:', case.isna().any(axis=1).sum())
print('Prezzi sotto 10 k€:', (case.prezzo < 10).sum())
print('Case oltre 500 m²:', (case.metri > 500).sum())

pulite = case.dropna()
pulite = pulite[(pulite.prezzo >= 10) & (pulite.metri <= 500)]
print('Righe rimaste:', len(pulite), 'su', len(case))

In [ ]:
plt.scatter(pulite.metri, pulite.prezzo, alpha=0.6, color='#1F3A5F')
plt.xlabel('metri quadri'); plt.ylabel('prezzo (k€)')
plt.show()

## 3. La retta migliore

Cerchiamo `prezzo = a + b × metri`.

### Metodo 1: la formula esatta (equazione normale)

In [ ]:
x = pulite.metri.values
y = pulite.prezzo.values

X = np.column_stack([np.ones_like(x), x])
a, b = np.linalg.solve(X.T @ X, X.T @ y)
print(f'prezzo = {a:.1f} + {b:.2f} × metri')

### Metodo 2: discesa del gradiente

Partiamo da una retta qualsiasi e la correggiamo a piccoli passi, sempre nella direzione in cui l'errore scende.
È lo stesso principio con cui si addestrano le reti neurali.

In [ ]:
# normalizziamo i metri, altrimenti i passi sono instabili
media, dev = x.mean(), x.std()
xn = (x - media) / dev

a_n, b_n = 0.0, 0.0
passo = 0.1
errori = []
for i in range(200):
    previsione = a_n + b_n * xn
    errore = previsione - y
    errori.append((errore ** 2).mean())
    a_n -= passo * 2 * errore.mean()
    b_n -= passo * 2 * (errore * xn).mean()

# riportiamo i coefficienti alla scala originale
b_gd = b_n / dev
a_gd = a_n - b_gd * media
print(f'prezzo = {a_gd:.1f} + {b_gd:.2f} × metri')

plt.plot(errori, color='#1F3A5F')
plt.xlabel('passo'); plt.ylabel('errore quadratico medio')
plt.show()

I due metodi arrivano alla stessa retta.

**Prova tu:** metti `passo = 1.2` e riesegui. Che cosa succede all'errore?

In [ ]:
xs = np.linspace(40, 200, 10)
plt.scatter(x, y, alpha=0.5, color='#1F3A5F')
plt.plot(xs, a + b * xs, color='#C25A00', linewidth=3)
plt.xlabel('metri quadri'); plt.ylabel('prezzo (k€)')
plt.show()

## 4. Dati mai visti

Un modello si valuta su dati che **non ha usato per imparare**.
Teniamo da parte il 25% delle case come test.
Con `scikit-learn` bastano tre righe.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    pulite[['metri', 'stanze']], pulite.prezzo, test_size=0.25, random_state=0)

modello = LinearRegression().fit(X_train, y_train)
rmse = mean_squared_error(y_test, modello.predict(X_test)) ** 0.5
print('Coefficienti:', modello.intercept_.round(1), modello.coef_.round(2))
print(f'Errore medio sul test: circa {rmse:.0f} k€')

In [ ]:
# quanto costa una casa di 90 m² con 3 stanze?
nuova = pd.DataFrame({'metri': [90], 'stanze': [3]})
print(f'{modello.predict(nuova)[0]:.0f} k€')

## 5. Overfitting

Prendiamo solo 12 case e confrontiamo:
- una retta (modello semplice);
- un polinomio di grado 11 (modello complicato).

In [ ]:
poche = pulite.sample(12, random_state=1).sort_values('metri')
resto = pulite.drop(poche.index)

def valuta(grado):
    coeff = np.polyfit(poche.metri / 100, poche.prezzo, grado)
    err_train = np.mean((np.polyval(coeff, poche.metri / 100) - poche.prezzo) ** 2) ** 0.5
    err_test = np.mean((np.polyval(coeff, resto.metri / 100) - resto.prezzo) ** 2) ** 0.5
    return coeff, err_train, err_test

xs = np.linspace(poche.metri.min(), poche.metri.max(), 300)
stili = {1: dict(color='#1F3A5F', linestyle='-', linewidth=3),
         11: dict(color='#C25A00', linestyle='--', linewidth=2)}
for grado in [1, 11]:
    coeff, tr, te = valuta(grado)
    print(f'grado {grado:2d}: errore sugli esempi {tr:8.1f}   errore su case nuove {te:12.1f}')
    plt.plot(xs, np.polyval(coeff, xs / 100), label=f'grado {grado}', **stili[grado])

plt.scatter(poche.metri, poche.prezzo, c='black', zorder=3)
plt.ylim(0, 450); plt.legend(); plt.show()

Il polinomio passa per tutti i punti (errore quasi zero sugli esempi) ma sulle case nuove sbaglia molto:
ha **imparato a memoria** invece di capire.

## Da ricordare

- Prima si puliscono i dati.
- Imparare = trovare i coefficienti che riducono l'errore.
- Il modello si giudica su dati nuovi, non su quelli che ha visto.
- *Tutti i modelli sono sbagliati, ma alcuni sono utili.*